# Procesamiento de datos

___

## Importaciones

In [30]:
import pandas as pd

df = pd.read_csv('../data/train.csv')

In [31]:
df

,id,Time_spent_Alone,Stage_fear,Social_event_attendance,Going_outside,Drained_after_socializing,Friends_circle_size,Post_frequency,Personality
0,0,0.0,No,6.0,4.0,No,15.0,5.0,Extrovert
1,1,1.0,No,7.0,3.0,No,10.0,8.0,Extrovert
2,2,6.0,Yes,1.0,0.0,NaN,3.0,0.0,Introvert
3,3,3.0,No,7.0,3.0,No,11.0,5.0,Extrovert
4,4,1.0,No,4.0,4.0,No,13.0,NaN,Extrovert
...,...,...,...,...,...,...,...,...,...
18519,18519,3.0,No,7.0,3.0,No,9.0,7.0,Extrovert
18520,18520,1.0,NaN,6.0,7.0,No,6.0,5.0,Extrovert
18521,18521,7.0,Yes,1.0,1.0,Yes,1.0,NaN,Introvert
18522,18522,NaN,Yes,1.0,0.0,Yes,5.0,2.0,Introvert


___

## Procesamiento

### Problemática actual

Tenemos una gran cantidad de datos faltantes en el dataset. Esta cantidad hace que perdamos casi el **40%** de nuestros datos al quitar las filas que contengan algún dato nulo.

Para esto vamos a realizar una **imputación de datos** para poder recuperar artificialmente este 40%. Realizar una imputación sencilla de datos haría que nuestro modelo no se entrenara correctamente, por lo que haremos un preprocesamiento de **clustering** para poder imputar los datos de manera más precisa, utilizando aquellos datos que se parezcan más entre ellos.

#### Pipeline
1. Reemplazamos las variables categóricas por numéricas
2. Creamos un dataset sin los datos nulos y los escalamos
3. Clusterizamos utilizando el dataset sin los datos nulos y agregamos la columna de cluster
4. Entrenamos un clasificador KNN con el dataframe y los clusters como etiquetas
5. Realizamos pruebas al clasificador
6. Creamos un dataset nuevo tomando como base el dataset original para hacerle una imputación simple y escalamos los datos
7. Utilizamos nuestro clasificador para predecir los clusters del dataset original utilizando como imput el dataframe que imputamos de manera simple
8. Exportamos el dataset para el entrenamiento del modelo

### 1. Reemplazamos las variables categóricas por variables numéricas

In [32]:
df['Stage_fear'] = df['Stage_fear'].map({'Yes': 1, 'No': 0})
df['Drained_after_socializing'] = df['Drained_after_socializing'].map({'Yes': 1, 'No': 0})

### 2. Creamos un dataset escalado sin los datos nulos

In [33]:
from sklearn.preprocessing import StandardScaler

features =  [
                'Time_spent_Alone', 'Stage_fear', 'Social_event_attendance',
                'Going_outside', 'Drained_after_socializing', 'Friends_circle_size',
                'Post_frequency'
            ]

df_notnull = df.dropna(subset=features)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_notnull[features])

### 3. Clusterizamos y creamos la columna de cluster

In [34]:
import hdbscan
clusterer = hdbscan.HDBSCAN(min_cluster_size=50)

df_notnull['cluster'] = clusterer.fit_predict(X_scaled)
df_notnull['cluster'].value_counts()

cluster
 1    8475
 0    1694
-1      20
Name: count, dtype: int64

Para este punto se esperaba encontrar una mayor cantidad de clusters y HDBSCAN únicamente encontró dos clusters, por lo que ahora surge la hipótesis de que tal vez los datos son tan descriptivos. Para intentar comprobar esta hipótesis vamos a ver lo siguiente:

1. Porcentaje de extrovertidos contra introvertidos en la clase objetivo
2. Porcentaje de puntos en cada cluster
3. Porcentajes cruzados entre los clusters y la clase objetivo

#### 3.1. Procentaje de extrovertidos e introvertidos en el dataset

In [35]:
df_notnull['Personality'].value_counts(normalize=True) * 100

Personality
Extrovert    82.657768
Introvert    17.342232
Name: proportion, dtype: float64

####  3.2. Porcentaje de puntos en cada cluster

In [36]:
df_notnull['cluster'].value_counts(normalize=True) * 100

cluster
 1    83.177937
 0    16.625773
-1     0.196290
Name: proportion, dtype: float64

#### 3.3. Porcentaje cruzado entre los clusters y la clase objetivo

In [37]:
pd.crosstab(df_notnull['cluster'], df_notnull['Personality'], normalize='index') * 100

Personality,Extrovert,Introvert
cluster,,
-1,45.000000,55.000000
0,8.677686,91.322314
1,97.533923,2.466077


### 4. Entrenamos un clasificador KNN

In [38]:
from sklearn.neighbors import KNeighborsClassifier

mask = df_notnull['cluster'] != -1
X_train_cluster = X_scaled[mask]
y_train_cluster = df_notnull.loc[mask, 'cluster']

knn_cluster = KNeighborsClassifier(n_neighbors=5)
knn_cluster.fit(X_train_cluster, y_train_cluster)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",5
,"weights weights: {'uniform', 'distance'}, callable or None, default='uniform'Weight function used in prediction. Possible values:- 'uniform' : uniform weights. All points in each neighborhood are weighted equally.- 'distance' : weight points by the inverse of their distance. in this case, closer neighbors of a query point will have a greater influence than neighbors which are further away.- [callable] : a user-defined function which accepts an array of distances, and returns an array of the same shape containing the weights.Refer to the example entitled:ref:`sphx_glr_auto_examples_neighbors_plot_classification.py`showing the impact of the `weights` parameter on the decisionboundary.",'uniform'
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float, default=2Power parameter for the Minkowski metric. When p = 1, this is equivalentto using manhattan_distance (l1), and euclidean_distance (l2) for p = 2.For arbitrary p, minkowski_distance (l_p) is used. This parameter is expectedto be positive.",2
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance<https://docs.scipy.org/doc/scipy/reference/spatial.distance.html>`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'minkowski'
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.Doesn't affect :meth:`fit` method.",None
Name,Type,Value
"classes_ classes_: array of shape (n_classes,)Class labels known to the classifier","ndarray[int64](2,)","[0,1]"
"effective_metric_ effective_metric_: str or callbleThe distance metric used. It will be same as the `metric` parameteror a synonym of it, e.g. 'euclidean' if the `metric` parameter set to'minkowski' and `p` parameter set to 2.",str,'eu...an'


### 5. Evaluamos el clasificador

In [39]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(knn_cluster, X_train_cluster, y_train_cluster, cv=5, scoring='accuracy')
print(f"Accuracy promedio: {scores.mean():.4f} (+/- {scores.std():.4f})")

Accuracy promedio: 1.0000 (+/- 0.0000)


### 6. Creamos un dataset con datos imputados de manera simple y predecimos los clusters en el dataset original

In [40]:
from sklearn.impute import SimpleImputer

df_temporal = df.copy()

imputer = SimpleImputer(strategy='median')
X_imputed = imputer.fit_transform(df_temporal[features])
X_imputed_scaled = scaler.transform(X_imputed)

df['cluster'] = knn_cluster.predict(X_imputed_scaled)
df

/home/fer/Repos/IntrovertsFromTheExtroverts/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2830: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


,id,Time_spent_Alone,Stage_fear,Social_event_attendance,Going_outside,Drained_after_socializing,Friends_circle_size,Post_frequency,Personality,cluster
0,0,0.0,0.0,6.0,4.0,0.0,15.0,5.0,Extrovert,1
1,1,1.0,0.0,7.0,3.0,0.0,10.0,8.0,Extrovert,1
2,2,6.0,1.0,1.0,0.0,NaN,3.0,0.0,Introvert,0
3,3,3.0,0.0,7.0,3.0,0.0,11.0,5.0,Extrovert,1
4,4,1.0,0.0,4.0,4.0,0.0,13.0,NaN,Extrovert,1
...,...,...,...,...,...,...,...,...,...,...
18519,18519,3.0,0.0,7.0,3.0,0.0,9.0,7.0,Extrovert,1
18520,18520,1.0,NaN,6.0,7.0,0.0,6.0,5.0,Extrovert,1
18521,18521,7.0,1.0,1.0,1.0,1.0,1.0,NaN,Introvert,0
18522,18522,NaN,1.0,1.0,0.0,1.0,5.0,2.0,Introvert,0


### 7. Realizamos una imputación final de los datos utilizando los clusters

In [41]:
from sklearn.impute import KNNImputer
import pandas as pd

df_final = df.copy()

for c in df['cluster'].unique():
    mask = df['cluster'] == c
    
    # 1. Tomar solo las filas de este cluster (con sus NaN reales todavía presentes)
    bloque = df.loc[mask, features]
    
    # 2. Escalar (mismo scaler ya ajustado con df_completo, solo transform)
    bloque_scaled = pd.DataFrame(
        scaler.transform(bloque),
        columns=features,
        index=bloque.index
    )
    
    # 3. Imputar en espacio escalado
    imputer = KNNImputer(n_neighbors=5)
    bloque_imputado_scaled = pd.DataFrame(
        imputer.fit_transform(bloque_scaled),
        columns=features,
        index=bloque.index
    )
    
    # 4. Regresar a la escala original
    bloque_imputado = pd.DataFrame(
        scaler.inverse_transform(bloque_imputado_scaled),
        columns=features,
        index=bloque.index
    )
    
    # 5. Escribir los valores imputados de vuelta en df_final
    df_final.loc[mask, features] = bloque_imputado

df_final[features] = df_final[features].round().astype(float)
df_final.drop(columns=['cluster'], inplace=True)

### 8. Exportamos el dataset para el entrenamiento del modelo

In [42]:
df_final.to_csv('../data/train_imputed.csv', index=False)

___

## Dado que al dataset de test también le faltan datos aplicamos el mismo pipeline

In [44]:
# Importamos el dataset de test
df_test = pd.read_csv("../data/test.csv")



# Cambiamos las variables categóricas a numéricas
df_test['Stage_fear'] = df_test['Stage_fear'].map({'Yes': 1, 'No': 0})
df_test['Drained_after_socializing'] = df_test['Drained_after_socializing'].map({'Yes': 1, 'No': 0})



# Creamos un nuevo dataset sin valores nulos
df_notnull = df_test.dropna(subset=features)



# Escalamos los datos y aplicamos HDBSCAN para obtener los clusters
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_notnull[features])

clusterer = hdbscan.HDBSCAN(min_cluster_size=50)

df_notnull['cluster'] = clusterer.fit_predict(X_scaled)
df_notnull['cluster'].value_counts()



# Entrenamos un clasificador KNN para predecir los clusters en el dataset de test
mask = df_notnull['cluster'] != -1
X_train_cluster = X_scaled[mask]
y_train_cluster = df_notnull.loc[mask, 'cluster']

knn_cluster = KNeighborsClassifier(n_neighbors=5)
knn_cluster.fit(X_train_cluster, y_train_cluster)



# Creamos un dataset temporal para imputar los valores faltantes en el dataset de test
df_temporal = df_test.copy()

imputer = SimpleImputer(strategy='median')
X_imputed = imputer.fit_transform(df_temporal[features])
X_imputed_scaled = scaler.transform(X_imputed)

df_test['cluster'] = knn_cluster.predict(X_imputed_scaled)



# Imputamos los valores faltantes en el dataset de test utilizando KNNImputer por cluster
df_final = df_test.copy()

for c in df_test['cluster'].unique():
    mask = df_test['cluster'] == c
    
    # 1. Tomar solo las filas de este cluster (con sus NaN reales todavía presentes)
    bloque = df_test.loc[mask, features]
    
    # 2. Escalar (mismo scaler ya ajustado con df_completo, solo transform)
    bloque_scaled = pd.DataFrame(
        scaler.transform(bloque),
        columns=features,
        index=bloque.index
    )
    
    # 3. Imputar en espacio escalado
    imputer = KNNImputer(n_neighbors=5)
    bloque_imputado_scaled = pd.DataFrame(
        imputer.fit_transform(bloque_scaled),
        columns=features,
        index=bloque.index
    )
    
    # 4. Regresar a la escala original
    bloque_imputado = pd.DataFrame(
        scaler.inverse_transform(bloque_imputado_scaled),
        columns=features,
        index=bloque.index
    )
    
    # 5. Escribir los valores imputados de vuelta en df_final
    df_final.loc[mask, features] = bloque_imputado

df_final[features] = df_final[features].round().astype(float)
df_final.drop(columns=['cluster'], inplace=True)
df_final.to_csv('../data/test_imputed.csv', index=False)

/home/fer/Repos/IntrovertsFromTheExtroverts/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2830: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
